# 03 — Análise de Resultados Tier 1

Carrega `results/tier1_results.json` e reproduz tabelas e figuras da dissertação.

Rode `python scripts/generate_analysis.py` para gerar todos os artefatos automaticamente.
Este notebook é para exploração interativa.

In [ ]:
import json
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.analysis.tables import results_table, sparsity_table, ranks_table
from src.analysis.plots import cd_diagram, boxplots, sparsity_accuracy_scatter, training_time_barplot
from src.metrics.statistical import friedman_test, average_ranks, nemenyi_cd, wilcoxon_pairwise

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

In [ ]:
# Carregar resultados
raw = json.loads(open('../results/tier1_results.json').read())
df = pd.DataFrame(raw)
df = df[df['status'] == 'ok'].copy()
df['model'] = df['model_variant'].fillna(df['model'])

LABELS = {
    'StandardLSSVM': 'LSSVM (Standard)', 'PCPLSSVm': 'LSSVM-PCP',
    'FSALSSVm': 'LSSVM-FSA', 'PruningLSSVM': 'LSSVM-Pruning',
    'IPLSSVm': 'LSSVM-IP', 'OppositeMapsLSSVM': 'LSSVM-OppMaps',
    'ADMMNesterovLSSVM': 'LSSVM-ADMM',
    'FTTransformer_softmax': 'FT-Softmax', 'FTTransformer_topk': 'FT-TopK',
    'FTTransformer_entmax': 'FT-Entmax', 'FTTransformer_sparsemax': 'FT-Sparsemax',
}
df['model_label'] = df['model'].map(LABELS).fillna(df['model'])

print(f"Runs: {len(df)} | Modelos: {df['model'].nunique()} | Datasets: {df['dataset'].nunique()}")
df.groupby('model_label')['f1_macro'].mean().sort_values(ascending=False).round(4)

## F1-macro por modelo e dataset

In [ ]:
pivot = df.groupby(['model_label', 'dataset'])['f1_macro'].mean().unstack().round(3)
pivot['MÉDIA'] = pivot.mean(axis=1)
pivot.sort_values('MÉDIA', ascending=False)

## Testes Estatísticos

In [ ]:
models = sorted(df['model'].unique())
datasets = sorted(df['dataset'].unique())

# Matriz (n_datasets, n_models) de médias por dataset
score_mat = np.full((len(datasets), len(models)), np.nan)
for j, m in enumerate(models):
    for i, d in enumerate(datasets):
        sub = df[(df['model'] == m) & (df['dataset'] == d)]['f1_macro']
        if len(sub): score_mat[i, j] = sub.mean()

friedman = friedman_test(score_mat)
print(f"Friedman: χ²={friedman['statistic']:.3f}, p={friedman['pvalue']:.6f}")

ranks = average_ranks(score_mat)
labels = [LABELS.get(m, m) for m in models]
cd = nemenyi_cd(len(models), len(datasets))
print(f"CD de Nemenyi (α=0.05): {cd:.3f}")

pd.DataFrame({'modelo': labels, 'rank_médio': ranks}).sort_values('rank_médio').reset_index(drop=True)

## Diagrama de Diferença Crítica

In [ ]:
fig = cd_diagram(ranks, labels, cd, 'Diagrama de Diferença Crítica — F1-macro')
plt.show()

## Boxplots por dataset

In [ ]:
df_display = df.copy()
df_display['model'] = df_display['model_label']
fig = boxplots(df_display.to_dict('records'), metric='f1_macro')
plt.show()

## Esparsidade × Performance

In [ ]:
fig = sparsity_accuracy_scatter(df_display.to_dict('records'))
plt.show()

## Tempo de Treino

In [ ]:
fig = training_time_barplot(df_display.to_dict('records'))
plt.show()

## Wilcoxon LSSVM-ADMM vs demais

In [ ]:
admm_scores = score_mat[:, models.index('ADMMNesterovLSSVM')]
results_wilcox = []
for j, m in enumerate(models):
    if m == 'ADMMNesterovLSSVM':
        continue
    other = score_mat[:, j]
    valid = ~(np.isnan(admm_scores) | np.isnan(other))
    if valid.sum() < 2:
        continue
    res = wilcoxon_pairwise(admm_scores[valid], other[valid])
    diff = admm_scores[valid].mean() - other[valid].mean()
    results_wilcox.append({
        'vs': LABELS.get(m, m),
        'diff_f1': round(diff, 4),
        'p_value': round(res['pvalue'], 4),
        'significativo': res['pvalue'] < 0.05
    })

pd.DataFrame(results_wilcox).sort_values('p_value')